# 决策树（Decision Tree）学习笔记
---
## 目录

1. [决策树概述与基本结构](#1-决策树概述与基本结构)
2. [训练与测试：难点在构造](#2-训练与测试难点在构造)
3. [信息熵：衡量混乱程度](#3-信息熵衡量混乱程度)
4. [信息增益：选节点的标准](#4-信息增益选节点的标准)
5. [构造实例：14 天打球完整推导](#5-构造实例14-天打球完整推导)
6. [三大算法：ID3 / C4.5 / CART](#6-三大算法id3--c45--cart)
7. [连续值处理](#7-连续值处理)
8. [剪枝：对抗过拟合](#8-剪枝对抗过拟合)
9. [决策树总结与速查](#9-决策树总结与速查)

---

## 1.  决策树概述与基本结构

**1.1 决策树概述**

* 决策树：从根节点开始，一步步走到叶子节点（决策）。所有数据最终都会落到叶子节点，既可以做分类，也可以做回归。

**1.2 树的组成**

| 组成部分 | 含义 | 通俗类比 |
|:---:|:---:|:---:|
| **根节点（Root Node）** | 第一个选择点（最顶层的判断） | 老大 |
| **非叶子节点与分支** | 中间过程（后续判断） | 二当家、三当家 |
| **叶子节点（Leaf Node）** | 最终的决策结果 | 最终落定的结论 |

**1.3 基本结构**
* 决策树是一种**树形结构**的模型，通过一系列"判断"做决策，类似于日常做决定的流程图。

```
                                                      [根节点：特征A的判断]
                                                      /                 \
                                                  [分支1]               [分支2]
                                                 /      \              /      \
                                          [叶子:类别1] [叶子:类别2]  [叶子:类别1] [叶子:类别3]
```
***

## 2. 训练与测试：难点在构造

**2.1 决策树的两个阶段**

| 阶段 | 内容 | 难点 |
|:---:|:---:|:---:|
| **训练阶段** | 从训练集构造出一棵树（从根节点开始选特征，确定切分） | **难点全在这里** |
| **测试阶段** | 根据构造好的树，从上到下走一遍 | 非常简单 |

* 一旦构造好了决策树，分类或预测就很简单了，走一遍即可。难点在于**如何构造一棵树**，需要考虑的问题很多。

**2.2 核心问题：如何切分特征**

* 问题：根节点该用哪个特征？接下来呢？如何切分？
* 目标：根节点要"像个老大"，能**更好地切分数据**，下面的节点依次是"二当家"
* 方法：通过一种**衡量标准**，计算不同特征分支后的分类情况，找**最好的那个当根节点**，以此类推
* 本质：是一个**贪心**过程：每一步选当前最优特征切分，递归构造整棵树。

---

## 3. 信息熵：衡量混乱程度

**3.1 熵的概念**

* 熵：随机变量不确定性的度量。说白了就是物体内部的混乱程度。

**3.2 熵的公式（可以画图理解）**

$$H(X) = -\sum_{i=1}^{n} p_i \log p_i$$

* 其中
    * $p_i$ 是第 $i$ 类别的概率，
    * $n$ 是类别总数。$\log$ 通常以 2 为底，
    * 单位为比特（bit）。

**3.3 熵的性质**

| 概率分布 | 熵值 | 含义 |
|:---:|:---:|:---:|
| $p = 0$ 或 $p = 1$ | $H(p) = 0$ | 完全没有不确定性（全是同一类） |
| $p = 0.5$ | $H(p) = 1$ | 不确定性最大（正负各半） |

* 规律：**不确定性越大，熵值越大。** 熵值 $H \in [0, \log n]$。

**3.4 一个栗子**

- A 集合：`[1,1,1,1,1,1,1,1,2,2]` → 只有 2 种类别，稳定 → **熵值低**
- B 集合：`[1,2,3,4,5,6,7,8,9,1]` → 类别太多 → **熵值大**

**3.5 分类任务中，我们希望节点分支后数据类别的熵值大还是小？**

* 答案：**希望熵值小**。分支后每堆越"纯"（同类聚在一起），说明切分得越好。

---

## 4. 信息增益：选节点的标准

**4.1  信息增益的定义**

* 定义：**信息增益 = 特征 X 使得类别 Y 的不确定性减少的程度。**
* 判断：**信息增益越大**，越能降低不确定性，越适合当节点

$$\text{Gain}(D, A) = H(D) - H(D \mid A)$$

* 其中：
    * $H(D)$：划分**前**整个数据集的熵
    * $H(D \mid A)$：按特征 $A$ 划分**后**的条件熵


**4.2  条件熵的计算**

$$H(D \mid A) = \sum_{v \in \text{Values}(A)} \frac{|D_v|}{|D|} \cdot H(D_v)$$

* 通俗理解：按特征 A 的每个取值把数据分成几堆，分别算每堆的熵，再按每堆占比加权平均。

---

## 5. 构造实例：14 天打球完整推导


**5.1 数据概况**

* 经典案例：**14 天打球情况**，4 个环境特征，目标构造决策树。
* 14 天：**9 天打球，5 天不打球**
* 4 个特征：Outlook、Temperature、Humidity、Windy

**5.2 第一步：根节点熵**

* 计算原始数据的熵
$$H(D) = -\frac{9}{14}\log_2\frac{9}{14} - \frac{5}{14}\log_2\frac{5}{14} = 0.940$$

**5.3 第二步：逐特征计算信息增益**
* 决策树有四种根节点划分方式，分别去计算他们的信息增益
    * 基于天气
    * 基于温度
    * 基于湿度
    * 基于有风情况

* 以 Outlook 为例（sunny / overcast / rainy）：

| Outlook 取值 | 打球:不打球 | 子集熵 |
|:---:|:---:|:---:|
| sunny | 2:3 | 0.971 |
| overcast | 4:0 | **0**（完全纯） |
| rainy | 3:2 | 0.971 |

* 计算条件熵：(概率：sunny = 5/14，overcast = 4/14，rainy = 5/14)
$$H(D \mid \text{Outlook}) = \frac{5}{14} \times 0.971 + \frac{4}{14} \times 0 + \frac{5}{14} \times 0.971 = 0.693$$

* 信息增益：

$$\text{Gain}(\text{Outlook}) = 0.940 - 0.693 = 0.247$$

**5.4 第三步：比较各特征增益**

| 特征 | 信息增益 |
|:---:|:---:|
| **Outlook** | **0.247（最大）** ✅ |
| Humidity | 0.152 |
| Windy | 0.048 |
| Temperature | 0.029 |

> **结论：Outlook 信息增益最大 → 当根节点（大当家）。** 然后对每个分支子集，继续用信息增益找二当家、三当家，递归构造，直到叶子纯净或满足停止条件。

---

## 6. 三大算法：ID3 / C4.5 / CART

| 算法 | 划分标准 | 特点 / 解决的问题 |
|:---|:---|:---|
| **ID3** | 信息增益 | 最经典，但**有缺陷**（见下） |
| **C4.5** | 信息增益率 | 解决 ID3 问题，**考虑自身熵** |
| **CART** | GINI 系数 | 分类 + 回归，生成二叉树 |

**6.1 ID3 的问题**

* 信息增益偏向于选择取值多的特征。
    * 例如"日期"特征，每个取值只对应一个样本，划分后每堆都是纯的，熵为 0，信息增益极大——但这毫无意义（过拟合，没有泛化能力）。

**6.2 C4.5 的信息增益率**

* 引入**信息增益率**，除以特征自身的熵来"惩罚"取值多的特征：

$$\text{GainRatio}(D, A) = \frac{\text{Gain}(D, A)}{H_A(D)}$$

* 其中
    * $H_A(D)$ 是特征 $A$ 自身的熵（$A$ 取值越多，$H_A$ 越大）。

**6.3 CART 的 GINI 系数**

$$Gini(D) = 1 - \sum_{i=1}^{n} p_i^2$$

- 与熵类似，衡量"纯度/混乱程度"
- 数据集越纯，GINI 越小
- 计算方式不同，思想相同；CART 每次**二分**

> 三者本质：**选择使划分后子集"最纯"的特征作为节点。**

---

## 7. 连续值处理

对连续特征（如温度 28.5℃），无法直接按离散取值划分。常用方法：

1. **二分法（CART 默认）**：
   - 连续值排序，取**每相邻两值的中间点**作为候选切分点
   - 对每个候选点，计算"≤ 该点 / > 该点"两堆的 GINI（或信息增益），选最优
2. **离散化**：把连续值分箱成若干区间（如低温/中温/高温）

> 连续特征每层可重复使用（在阈值两侧继续二分），这是与离散特征的重要区别。

---

## 8. 剪枝：对抗过拟合

**8.1 为什么要剪枝**

* 决策树**过拟合风险很大**，理论上可完全分得开数据——树足够庞大时，每个叶子节点就只剩一个数据了。

* 完全拟合训练数据 → 泛化差 → 必须剪枝。

**8.2 两种剪枝策略**

| 策略 | 时机 | 做法 | 评价 |
|:---|:---|:---|:---|
| **预剪枝** | 边建树边剪 | 限制深度、叶子节点数、叶子节点样本数、信息增益量 | **更实用** |
| **后剪枝** | 建完树再剪 | 通过衡量标准（叶子节点越多，损失越大）裁剪 | 效果更好但开销大 |

**8.3 预剪枝常见超参数：**

- 树的最大深度（max_depth）
- 叶子节点个数上限
- 叶子节点最少样本数（min_samples_leaf）
- 信息增益阈值（低于就不继续分）

> 正则化思想：限制模型复杂度，防止"每个叶子只装一个样本"。

**8.3 后剪枝：**
* 通过一定的衡量标准（叶子节点越多，$C_a(T)$越大，越不好）
  
$$C_a(T) = C(T) + a\,|T_{leaf}|$$

* 其中：
    * $C(T)$: 自身的gini系数或熵值*sample个数
    * $a\,|T_{leaf}|$: 从他开始的叶子节点数

---

## 9. 决策树求解回归问题

**9.1 回归树vs分类树**

* 分类树追求"子集越**纯**越好"，回归树追求"子集内**离均值越近**越好"。

| 维度 | 分类树 | 回归树 |
|:---|:---|:---|
| **目标变量** | 离散类别（如 0/1、多类） | 连续值（如房价、额度） |
| **叶子节点输出** | 类别（多数投票） | **该叶子所有样本的均值** |
| **衡量切分好坏** | 熵 / 信息增益 / GINI（纯度） | **平方误差之和**（误差） |
| **典型算法** | ID3 / C4.5 / CART 分类 | **CART 回归树** |

**9.2 损失函数：平方误差**

* 回归树的切分标准是**最小化平方误差**，这与线性回归的最小二乘思想一脉相承。

* 对于某个区域（节点）$R$，其内部误差为：

$$\text{MSE}(R) = \sum_{x_i \in R} (y_i - \hat{y}_R)^2$$

* 其中
    * $\hat{y}_R$ 是该区域所有样本标签的**均值**：

$$\hat{y}_R = \frac{1}{|R|} \sum_{x_i \in R} y_i$$

> 为什么叶子取值是均值？因为对平方误差而言，**均值就是使平方误差最小的唯一解**（对 $\hat{y}$ 求导令其为 0 可得）。这与线性回归中"最小二乘解是均值/线性拟合"是同一个道理。

**9.3 如何选择切分变量与切分点（核心）**

* 回归树要同时决定两件事：**用哪个特征切**、**在哪个点切**。目标是让切分后的**总平方误差最小**。

* 目标函数：对特征 $j$ 和切分点 $s$，把数据分成两个区域：
    * $R_1(j, s) = \{ x \mid x^{(j)} \le s \}$（左子节点）
    * $R_2(j, s) = \{ x \mid x^{(j)} > s \}$（右子节点）

求解：

$$\min_{j, s} \left[ \sum_{x_i \in R_1(j,s)} (y_i - \bar{y}_{R_1})^2 + \sum_{x_i \in R_2(j,s)} (y_i - \bar{y}_{R_2})^2 \right]$$

其中 $\bar{y}_{R_1}$、$\bar{y}_{R_2}$ 分别是左右两个区域的样本均值。

**9.3.2 遍历搜索**

* 由于特征通常是连续值，回归树采用**遍历搜索**：

```
1. 对每个特征 j：
   a. 将该特征的所有取值排序
   b. 取相邻两个值的中间点作为候选切分点 s
   c. 对每个候选 s，计算切分后的总平方误差
2. 选出使总平方误差最小的 (j, s) 组合
3. 用最优 (j, s) 切分当前节点
```

> 与分类树处理连续值一样，回归树天然适合连续特征，无需额外离散化。

**9.4 叶子节点的输出值**

* 回归树构造完成后，**每个叶子节点的预测值 = 该叶子内所有训练样本标签的均值**。

$$预测值 = \frac{1}{n_{\text{leaf}}} \sum_{i \in \text{leaf}} y_i$$

> 对一个新的测试样本，从根节点按切分条件一路走到叶子，直接输出该叶子的均值即可——**测试阶段与分类树一样简单**。

**9.5 完整算法流程（CART 回归树）**

```
输入：训练集 D，停止条件
1. 若满足停止条件（节点样本数过少 / 深度达到上限 / 误差足够小）：
   将该节点设为叶子，输出其样本均值
2. 否则：
   a. 遍历所有特征 j 和所有候选切分点 s
   b. 找使总平方误差最小的 (j, s)
   c. 按 x^(j) ≤ s 和 x^(j) > s 把数据分成两个子节点
   d. 对两个子节点递归执行步骤 1-2
3. 返回整棵回归树
```

---

## 9.6 一个简单实例

用「面积 → 房价」演示回归树的切分思路：

| 面积 x | 房价 y（万） |
|:---:|:---:|
| 10 | 50 |
| 20 | 70 |
| 30 | 100 |
| 40 | 140 |
| 50 | 190 |
| 60 | 250 |

**候选切分点**（相邻面积的中点）：15、25、35、45、55

以切分点 **35**（即 $x \le 35$ 与 $x > 35$）为例：

- 左区域（10,20,30）：均值 = (50+70+100)/3 = 73.3
  - 平方误差 = (50−73.3)² + (70−73.3)² + (100−73.3)² ≈ 1266.7
- 右区域（40,50,60）：均值 = (140+190+250)/3 = 193.3
  - 平方误差 = (140−193.3)² + (190−193.3)² + (250−193.3)² ≈ 6066.7

总误差 ≈ 7333.3

遍历所有候选切分点，取总误差**最小**的那个作为根节点切分点，然后对左右子节点递归重复。最终叶子输出各自均值，得到一棵分段常数（阶梯状）的回归树。

> 直观理解：回归树得到的是一条**分段常数的阶梯曲线**，通过不断切分来逼近真实的非线性关系。

---

## 9.7 决策树回归的优缺点

| 优点 | 缺点 |
|:---|:---|
| 能拟合**非线性关系**（无需假设线性） | 容易**过拟合**，必须剪枝 |
| 无需特征缩放/标准化 | 对**异常值敏感** |
| 可解释性好（可视化决策规则） | 输出是**分段常数**，不连续 |
| 天然处理连续特征 | 高维稀疏数据表现一般 |

---

## 10. 决策树总结与速查

**10.1 核心流程**

```
1. 计算当前节点数据集的熵 H(D)（分类）或平方误差（回归）
2. 对每个特征计算信息增益（或增益率/GINI/平方误差减少量）
3. 选增益最大的特征（分类）或误差最小的切分（回归）
4. 按该特征/切分点分裂出子节点
5. 对每个子节点递归重复 1-4
6. 直到满足停止条件 → 剪枝
```

**10.2 关键公式速查**

| 名称 | 公式 |
|:---|:---|
| 熵 | $H(X) = -\sum_{i=1}^{n} p_i \log p_i$ |
| 条件熵 | $H(D \mid A) = \sum_v \frac{\|D_v\|}{\|D\|} H(D_v)$ |
| 信息增益 | $\text{Gain}(D, A) = H(D) - H(D \mid A)$ |
| 信息增益率 | $\text{GainRatio} = \frac{\text{Gain}(D,A)}{H_A(D)}$ |
| GINI 系数 | $Gini(D) = 1 - \sum_{i=1}^{n} p_i^2$ |
| 回归树切分 | $\min_{j,s} \left[ \sum_{R_1}(y-\bar{y}_{R_1})^2 + \sum_{R_2}(y-\bar{y}_{R_2})^2 \right]$ |
| 回归叶子输出 | $\hat{y}_{\text{leaf}} = \frac{1}{n}\sum_{i \in \text{leaf}} y_i$ |

**10.3 分类树三大算法对比**

| 维度 | ID3 | C4.5 | CART |
|:---|:---|:---|:---|
| 划分标准 | 信息增益 | 信息增益率 | GINI 系数 |
| 树类型 | 多叉树 | 多叉树 | 二叉树 |
| 连续值 | 不支持 | 支持 | 支持 |
| 任务 | 分类 | 分类 | 分类 + 回归 |
| 缺点 | 偏向取值多的特征 | 计算复杂 | 每次二分 |

**10.4 分类树 vs 回归树**

| 维度 | 分类树 | 回归树 |
|:---|:---|:---|
| 目标变量 | 离散类别 | 连续值 |
| 切分标准 | 熵/信息增益/GINI | 平方误差最小 |
| 叶子输出 | 类别（多数投票） | 样本均值 |
| 代表算法 | ID3/C4.5/CART分类 | CART回归树 |

**10.5 一句话总结**

> **决策树 = 用衡量标准（分类用纯度、回归用平方误差）递归选特征切分数据，构造从根到叶的判定树；为防过拟合需剪枝。** 难点全在"如何构造树"。